# Week 3 — Data Contract and Feature Frame

**Author:** Zain-ul-Abdeen
**Lane:** Lane 4 — CTR / Engagement Opportunity Scoring
**Assignment:** ML-03

## 1. The Contract (Plain Words)

1. **What one row means for my lane:** One row represents a single piece of content (`content_hash_id`) for a specific client (`client_hash_id`) aggregated over a specific trailing 30-day window.
2. **Which table(s) I'll use:** `fact_content_daily_performance` (for daily impressions, clicks, position, and GA4 metrics) and `fact_content_query_90d` (for query concentration).
3. **Which time window:** A mid-panel month, specifically `month=2026-03`.
4. **What I'd predict or rank:** I will rank pages by their **CTR Opportunity Gap** (`expected_ctr` vs actual `ctr_30d`) scaled by search volume.
5. **One thing I deliberately exclude:** I deliberately exclude the `trend_direction` label and any performance data from the period *after* the decision moment (e.g. `month=2026-04`) to prevent time-travel leakage.

In [1]:
import os, getpass
import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

# Authenticate with Hugging Face
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("DuckDB connected to FlyRank warehouse on Hugging Face.")

Paste your Hugging Face READ token (hf_...): ··········
DuckDB connected to FlyRank warehouse on Hugging Face.


## 2. Prove Three Facts (Querying month=2026-03)

In [2]:
# Fact 1: Grain verification (one row = client x content x date)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as row_count
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
""").df()

print(f"Grain check violations (expected 0): {len(grain_check)}")
if len(grain_check) > 0:
    print(grain_check.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check violations (expected 0): 0


In [3]:
# Fact 2: Row count and date span for our slice
span_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("Row count and date span:")
print(span_check)

Row count and date span:
   total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


In [4]:
# Fact 3: Availability (GA4 metrics filter)
availability_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
""").df()

print("Availability (rows surviving IS TRUE filter for GA4):")
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability (rows surviving IS TRUE filter for GA4):
   total_rows  ga4_available_rows
0     9841378            413966.0


## 3. Five Features and The Trap

**The 5 Features for my Lane:**
1. `impressions_30d`: knowable at the decision moment because it's aggregated from the trailing 30 days of Search Console data.
2. `clicks_30d`: knowable at the decision moment because it's exactly the observed historical clicks before making a decision.
3. `avg_position_30d`: knowable at the decision moment because it reflects the average ranking observed in the trailing 30-day window.
4. `sessions_30d`: knowable at the decision moment because GA4 sessions from the trailing window are already logged.
5. `ctr_30d`: knowable at the decision moment because it is derived directly from `clicks_30d` and `impressions_30d`.

In [5]:
# Build the feature frame for month=2026-03
# Note: downloading remote columns takes ~2 minutes
features_query = f"""
    WITH march_data AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_30d,
               SUM(gsc_clicks) AS clicks_30d,
               AVG(gsc_avg_position) AS avg_position_30d,
               SUM(ga4_sessions) AS sessions_30d
        FROM {TABLES['fact_daily']}
        WHERE report_date BETWEEN '2026-03-01' AND '2026-03-31'
          AND ga4_data_available IS TRUE
        GROUP BY 1, 2
        HAVING SUM(gsc_impressions) >= 100
    )
    SELECT * FROM march_data
"""
features = con.sql(features_query).df()
features['ctr_30d'] = (features['clicks_30d'] / features['impressions_30d'] * 100).fillna(0)

print(f"Feature frame shape: {features.shape[0]:,} rows, {features.shape[1]} columns")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: 32,596 rows, 7 columns


,client_hash_id,content_hash_id,impressions_30d,clicks_30d,avg_position_30d,sessions_30d,ctr_30d
0,client_9958f0a7ae1df715,content_810cf06597918291,257.0,1.0,11.186123,42.0,0.389105
1,client_9958f0a7ae1df715,content_b813c73d7000b3b1,180.0,1.0,8.674734,7.0,0.555556
2,client_9958f0a7ae1df715,content_5a77dbf5671c5a65,19657.0,199.0,4.532382,168.0,1.012362
3,client_9958f0a7ae1df715,content_278030b007943b07,319.0,7.0,5.914484,16.0,2.194357
4,client_9958f0a7ae1df715,content_237e63fc00c8c7ad,6106.0,39.0,7.108246,40.0,0.638716


### The Trap (Feature Leakage)

If I were building a regression model to predict `ctr_30d` or clicks, and I accidentally included a target-derived metric from the *same* window (or a future window), my model's performance would jump to near-perfect R-squared. Let's see this in action by leaking `clicks_30d` (the target proxy) into the feature set as a "derived engagement score."

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

# THE TRAP: Adding a label-derived column on purpose
features['magic_engagement_score'] = features['clicks_30d'] * 1.5 + np.random.normal(0, 1, len(features))

# Our target is ctr_30d
X_leaky = features[['impressions_30d', 'avg_position_30d', 'sessions_30d', 'magic_engagement_score']]
y = features['ctr_30d']

X_train, X_test, y_train, y_test = train_test_split(X_leaky, y, test_size=0.2, random_state=42)

leaky_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
leaky_model.fit(X_train, y_train)
preds_leaky = leaky_model.predict(X_test)
leaky_score = r2_score(y_test, preds_leaky)

print(f"Leaky Model R^2 Score (with the trap feature): {leaky_score:.3f}")

# REMOVING THE LEAK
X_honest = features[['impressions_30d', 'avg_position_30d', 'sessions_30d']]
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_honest, y, test_size=0.2, random_state=42)

honest_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
honest_model.fit(X_train_h, y_train_h)
preds_honest = honest_model.predict(X_test_h)
honest_score = r2_score(y_test_h, preds_honest)

print(f"Honest Model R^2 Score (without the trap feature): {honest_score:.3f}")

# Cleaning up the dataframe
features = features.drop(columns=['magic_engagement_score'])

Leaky Model R^2 Score (with the trap feature): 0.914
Honest Model R^2 Score (without the trap feature): 0.199


## 4. One Named Limitation

**Limitation of this slice:** We lack visibility into SERP features (like Featured Snippets, Knowledge Panels, or AI Overviews). A page might have a low CTR despite a great `avg_position` simply because the search engine answered the user's query directly on the SERP (a zero-click query). Our model cannot distinguish between a "bad meta description" and a "zero-click SERP" using only aggregate GSC metrics.

## 5. Self-Check

| Check | Answer |
|---|---|
| **Contract answers in plain words?** | Yes, all 5 answered (grain, tables, window, target proxy, excluded feature). |
| **Exactly three verification queries?** | Yes, grain check, date span check, and IS TRUE availability check on month=2026-03. |
| **Outputs visible?** | Yes. |
| **Five-feature frame with 'knowable' lines?** | Yes, 5 features mapped and explained. |
| **Trap shown and removed?** | Yes, the `magic_engagement_score` leakage trap was built, tested (artificially high R^2), and dropped. |
| **One named limitation?** | Yes, lack of SERP feature visibility. |